새로운 api로 실습 진행

In [ ]:
ls ./drive/MyDrive/api*

./drive/MyDrive/apikeys.txt


In [ ]:
# API 키 파일을 읽어서 환경변수로 등록하는 코드
from dotenv import load_dotenv
load_dotenv('./drive/MyDrive/apikeys.txt')

True

In [ ]:
more ./drive/MyDrive/apikeys.txt

In [ ]:
# openai 라이브러리에서 OpenAI 불러오기
from openai import OpenAI

In [ ]:
client = OpenAI()

In [ ]:
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"

In [ ]:
messages = [
    {
        'role':'user', 'content' : [{'type':'text', 'text':'이미지를 설명해 주세요.'},
    {'type':'image_url', 'image_url':{'url':image_url}}]
    }
]

In [ ]:
response=client.chat.completions.create(model='gpt-4o-mini',messages=messages)
print(response.choices[0].message.content)

이미지에는 두 마리의 고양이가 핑크색 소파 위에서 편안하게 잠을 자고 있습니다. 왼쪽에 있는 고양이는 길고 두꺼운 줄무늬가 있고, 오른쪽의 고양이는 더 짧은 줄무늬와 동그란 얼굴을 가지고 있습니다. 두 고양이 모두 각자 다른 자세로 눕고 있으며, 소파 위에는 여러 개의 리모컨이 놓여 있습니다. 전체적인 분위기는 아늑하고 편안해 보입니다.


In [ ]:
import json


def get_current_weather(location, unit="Celsius"):
    if "seoul" in location.lower():
        return json.dumps({"location": "Seoul", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps(
            {"location": "San Francisco", "temperature": "72", "unit": unit}
        )
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

In [ ]:
get_current_weather('seoul')

'{"location": "Seoul", "temperature": "10", "unit": "Celsius"}'

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

In [ ]:
client = OpenAI()

In [ ]:
messages = [{
    'role':'user','content':'서울 날씨는 어떤가요?'}]

In [ ]:
response = client.chat.completions.create(model='gpt-4o', messages=messages, tools = tools)

In [ ]:
response_message = response.choices[0].message

In [ ]:
response_message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_AbxbPPsohjHAEEc7MREO9xmJ', function=Function(arguments='{"location":"Seoul, South Korea","unit":"celsius"}', name='get_current_weather'), type='function')])

In [ ]:
response_message.to_dict()

{'content': None,
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'tool_calls': [{'id': 'call_AbxbPPsohjHAEEc7MREO9xmJ',
   'function': {'arguments': '{"location":"Seoul, South Korea","unit":"celsius"}',
    'name': 'get_current_weather'},
   'type': 'function'}]}

- tool_calls : 도구 호출 요청  
- LLM이 함수 요청을 보내고 끝내버림  
-> LLM이 자기가 할 일은 여기까지라 판단 후 이제 실행은 인간(or 시스템)이 해야 한다는 뜻



```
{'content': None,
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'tool_calls': [{'id': 'call_mEhQUdTBKrPPtsy2Uu3F1q0g',
   'function': {'arguments': '{"location":"Seoul, South Korea"}',
    'name': 'get_current_weather'},
   'type': 'function'}]}
```
이 출력은 LLM이 말 대신 함수 실행을 요청하고 멈춘 상태를 보여줌  
-> 텍스트로 답하지 말고, get_current_weather 함수를 인자로 실행하라고 요청한 것  
=>
- content가 None
- 대신 tool_calls가 채워짐


#### <올바른 흐름>
1. 사용자 질문  
-> 서울 날씨 알려줘
2. LLM 판단  
-> 이건 날씨 API가 필요하네 -> get_current_weather 호출해야 겠다
3. LLM 응답  
-> too_calls=[] -> 여기서 LLM 멈춤
4. 인간(or 시스템)이 해야 할 일  
-> 실제로 함수 실행
5. 함수 실행 결과를 다시 LLM에 전달  
6. LLM이 다시 말함  
-> 현재 서울 날씨는 맑고 25도 입니다

In [ ]:
messages

[{'role': 'user', 'content': '서울 날씨는 어떤가요?'}]

In [ ]:
messages.append(response_message.to_dict())

In [ ]:
messages

[{'role': 'user', 'content': '서울 날씨는 어떤가요?'},
 {'content': None,
  'refusal': None,
  'role': 'assistant',
  'annotations': [],
  'tool_calls': [{'id': 'call_AbxbPPsohjHAEEc7MREO9xmJ',
    'function': {'arguments': '{"location":"Seoul, South Korea","unit":"celsius"}',
     'name': 'get_current_weather'},
    'type': 'function'}]}]

In [ ]:
available_functions = {
    "get_current_weather": get_current_weather,
}

# 사용하고 싶은 함수는 여러 개일 수 있으므로 반복문 사용
for tool_call in response_message.tool_calls:
    # 함수를 실행
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(
        location=function_args.get("location"),
        unit=function_args.get("unit"),
    )
    print(function_response)

    # 함수 실행 결과를 대화 이력으로 messages에 추가
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

{"location": "Seoul", "temperature": "10", "unit": "celsius"}


In [ ]:
second_response = client.chat.completions.create(model='gpt-4o', messages=messages)

In [ ]:
second_response.choices[0].message.content

'현재 서울의 기온은 약 10도입니다.'

In [ ]:
# LLM이 사용할 수 있는 함수(도구) 등록하는 코드
from langchain.tools import tool # LangChain에서 제공하는 @tool 데코레이터를 가져옴
                                 # 데코레이터 -> 그냥함수를 LLM이 쓰는 도구로 바꿈

@tool #바로 아래 함수는 LLM이 호출 가능한 도구이다
# 이 함수는 LLM이 함수를 사용해야 겠다고 판단할 때만 사용됨
def multiply(a: int, b: int) -> int:
  """두 숫자를 곱합니다."""
  return a*b

In [ ]:
# 파이썬 환경에 OpenAI 모델 연동 모듈 설치
# pip install langchain_openai
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 4.1 MB/s eta 0:00:00


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = "필요하면 계산 도구를 사용하세요."

agent = create_agent(model=llm, tools=[multiply], system_prompt=prompt)

inputs = {"messages": [{"role": "user", "content": "6 곱하기 9는 얼마야?"}]}

result = agent.invoke(inputs)

print(result)

{'messages': [HumanMessage(content='6 곱하기 9는 얼마야?', additional_kwargs={}, response_metadata={}, id='cea919d1-c6f3-4dd0-9f75-a2604a7ab1ed'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 67, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_aa07c96156', 'id': 'chatcmpl-CnGhBa4OQhWjgvpXt75TNGas9PF79', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b255b-15dd-75c0-9f2f-f9afc09c317c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 6, 'b': 9}, 'id': 'call_4lF8hOXQAi7o4EAdAoIhbrvB', 'type': 'tool_call'}], usage_metadata={'input_tokens': 67, 'output_tokens': 17, 'total_tokens': 84, 'input_toke

In [ ]:
result['messages']

In [ ]:
len(result['messages'])

In [ ]:
result['messages'][-1].content

In [ ]:
# 함수를 다른 함수로 감싸서 실행 전/후에 동작을 추가하는 구조 -> 데코레이터의 기본 형태
def wrapper(func): # 외부함수
  def wrapped_func(): # 내부함수
    print('=====before=====')
    func()  # 외부 함수를 내부 함수가 끌어다 쓰고 있음(참조)
    print('=====after=====')
  return wrapped_func  # 외부함수가 내부함수를 반환함

def myfunc():
  print('     I am here.')

#### < 데코레이터 조건>
1. 함수가 중첩되어 있어야 함
2. 내부함수가 외부함수를 참조하고 있음
3. 외부함수가 내부함수를 반환함

In [ ]:
result = wrapper(myfunc)




```
result = wrapper(myfunc)
```
위 코드는


```
def wrapped_func()
  print("=========")
  myfunc()
  print("=========")
```
이런 구조를 만듦
  

In [ ]:
result()

In [ ]:
@wrapper  # 아래 정의된 myfunc2() 함수를 wrapper(myfunc2) 형태로 자동 넘김
def myfunc2():
  print('         me too')

# 파이썬은 내부적으로 이렇게 실행함
# def myfunc2():
#   print('       me too')
# myfunc2 = wrapper(myfunc2)

In [ ]:
myfunc2()